# Módulo 3 — 1/3 - APIs y servicios REST

En esta clase vamos a cubrir:

1. Qué es una API y qué tipos existen
2. JSON y diccionarios de Python: el mismo dato en dos formatos
3. Recursos, endpoints y los verbos HTTP
4. Códigos de estado: qué significa cada uno y qué hace tu script
5. La librería `requests`: llamadas reales y el CRUD completo
6. `timeout` y parseo de respuestas XML
7. Postman como herramienta de depuración
8. Autenticación: API keys, OAuth y JWT
9. Rate limiting y gestión de errores

> Las secciones que llaman a Internet usan `https://jsonplaceholder.typicode.com`, una API
> pública y gratuitade pruebas. Si no tenés conexión, la celda te avisa y seguís con el resto.

Comprobamos que el intérprete es el correcto y que tenemos lo necesario instalado.

In [ ]:
import sys

print("Python:", sys.executable)
print("Version:", sys.version.split()[0])

In [ ]:
import importlib

for paquete in ("requests", "jwt", "schedule"):
    try:
        importlib.import_module(paquete)
        print(f"{paquete:10} OK")
    except ImportError:
        print(f"{paquete:10} FALTA  ->  pip install {paquete}")

---

## 1. ¿Qué es una API?

Una **API** (Application Programming Interface) es el conjunto de reglas que permite que
un programa parle con otro sin conocer sus detalles internos. Cuando consultás el clima desde
una app, esa app no lee nada del satélite: le pide el dato a la API meteorológica y dibuja lo
que le devuelven.

Piénsalo como el mostrador de un restaurante: vos (el cliente) no entrás a la cocina, pedís y
recibís. El **menú es la API**: define qué se puede pedir, con qué formato y qué recibís.

| Tipo | Cómo funciona | Cuándo lo encontrás |
|------|---------------|---------------------|
| **REST** | Sobre HTTP, con recursos y verbos. Flexible y mayoritario | APIs públicas, móviles, web |
| **SOAP** | Sobre HTTP, pero con un contrato XML muy rígido | Sistemas legacy, banca,_some XML estricto |
| **GraphQL** | El cliente pide exactamente los campos que necesita | Facebook, GitHub, apps con muchos datos |

Los tres tienen el mismo objetivo: que dos sistemas distintos se entiendan sin acoplarse.

In [ ]:
# Comparativa como dato: sirve para imprimirla, para tests o para generar documentacion
TIPOS_DE_API = {
    "REST":   "Sobre HTTP, con recursos y verbos. Flexible y mayoritario.",
    "SOAP":   "Sobre HTTP, con un contrato XML muy rigido.Tipico de banca y sistemas legacy.",
    "GraphQL": "El cliente pide exactamente los campos que necesita, en una sola peticion.",
}

for nombre, descripcion in TIPOS_DE_API.items():
    print(f"{nombre:8} {descripcion}")

En este módulo trabajamos con **REST**, porque es lo que vas a encontrar en cualquier
web, app o servicio que consumas con `requests`.

---

## 2. JSON y diccionarios de Python

**JSON** es un formato de texto para intercambiar datos: pares clave-valor, jerarquía
anidada y cuatro tipos (texto, número, booleano y `null`). Es legible por humanos y
fácil de leer por máquinas.

Un **diccionario** de Python tiene exactamente la misma forma: pares clave-valor y anidación.
Por eso el módulo `json` de la biblioteca estándar traduce entre los dos en ambos sentidos:

| Operación | Función | Resultado |
|-----------|---------|-----------|
| Python → JSON | `json.dumps(dict)` | cadena de texto |
| JSON → Python | `json.loads(cadena)` | `dict`, `list`, `str`... |
| Archivo → Python | `json.load(archivo)` | el objeto del archivo |
| Python → archivo | `json.dump(obj, archivo)` | escribe el archivo |

Es una calle de doble sentido: la misma información, en el formato que cada lado entiende.

In [ ]:
import json

persona = {"name": "Alice", "age": 30}

# Python -> JSON (serializar)
texto = json.dumps(persona)
print("json.dumps ->", texto)

# JSON -> Python (cargar de nuevo)
recuperada = json.loads(texto)
print("json.loads ->", recuperada)

# Ida y vuelta: los datos originales se conservan
print("Es el mismo dict:", recuperada == persona)

Un detalle que sorprende la primera vez: `json.dumps` mete un espacio después de los dos
puntos, así que el texto **no es idéntico** al que verías en un ejemplo de la documentación.
No importa para nada: lo que cuenta es que al volver a cargarlo obtenemos el mismo objeto.

In [ ]:
# indent=2 hace el JSON legible por humanos; sin esto, va todo en una linea
print(json.dumps(persona, indent=4))


In [ ]:

# ensure_ascii=False conserva los acentos y la ñ en vez de escaparlos como \u00f1
con_tilde = {"ciudad": "Córdoba", "pais": "Argentina"}
print("Por defecto:", json.dumps(con_tilde))


In [ ]:

print("Sin escapes: ", json.dumps(con_tilde, ensure_ascii=False))

El anidamiento funciona igual que en cualquier diccionario: un valor puede ser otro
diccionario, o una lista de diccionarios. Es la forma habitual en que vienen los JSON de una
API.

In [ ]:
# Un solo usuario de https://jsonplaceholder.typicode.com/users/1
usuario = {
    "id": 1,
    "name": "Leanne Graham",
    "address": {
        "street": "Kulas Light",
        "city": "Gwenborough",
        "geo": {"lat": "-37.3159", "lng": "81.1496"},
    },
    "company": {"name": "Romaguera-Crona"},
}

print("Nombre:", usuario["name"])
print("Ciudad:", usuario["address"]["city"])
print("Latitud:", usuario["address"]["geo"]["lat"])
print("Empresa:", usuario["company"]["name"])

# .get() con valor por defecto: no rompe si la clave no existe
print("Telefono:", usuario.get("phone", "no informado"))
print("Codigo postal:", usuario["address"].get("zipcode", "no informado"))

### La trampa de la mutabilidad

Los diccionarios son mutables. Si asignas uno a otro nombre, **ambos nombres apuntan al mismo
objeto**. Alguien que lo modifique thinking "este es el suyo" en realidad te cambia a ti el
dato. Con una configuración por defecto esto se convierte en un bug muy difícil de ver.

La regla: copia explícitamente cuando quieras un valor independiente.

In [ ]:
AJUSTES_POR_DEFECTO = {
    "idioma": "es",
    "notificaciones": {"email": True, "sms": False},
}

# MAL: dos variables, un solo objeto
mal = AJUSTES_POR_DEFECTO
mal["notificaciones"]["email"] = False

print("Mal:", mal)

print("El original tambien cambio:", AJUSTES_POR_DEFECTO)


Para hacer copias completas de objetos python nos ofrece la libreria deepcopy

In [ ]:
from copy import deepcopy

In [ ]:
AJUSTES_POR_DEFECTO = {
    "idioma": "es",
    "notificaciones": {"email": True, "sms": False},
    "tema": None
}

# BIEN: copia profunda, cada quien con la suya
bien = deepcopy(AJUSTES_POR_DEFECTO)
bien["notificaciones"]["email"] = False
bien["idioma"] = "en"

print(bien)

print("El original quedo intacto:", AJUSTES_POR_DEFECTO)

In [ ]:
print(json.dumps(AJUSTES_POR_DEFECTO, indent=4, ensure_ascii=False))

In [ ]:
var= json.loads("""{
    "idioma": "es",
    "notificaciones": {
        "email": true,
        "sms": false
    },
    "tema": null
}""")

In [ ]:
type(var)  # dict

In [ ]:
var

### Dónde aparece JSON en la vida real

- **Respuestas de APIs**: es lo que devuelve casi todo lo que veremos a partir de la sección 5.
- **Archivos de configuración**: `settings.json`, `config.json`.
- **Análisis de datos**: `pandas` lee JSON directo.
- **Logs y exportaciones**: Nagios, Prometheus, herramientas de diseño.

Ojo: JSON **no tiene esquema nativo**. Nada obliga a que las claves existan, así que un dato
que ayer venía y hoy no, rompe el script. Validar antes de usar es trabajo tuyo.

In [ ]:
import pandas as pd

catalogo = [
    {"producto": "Teclado",   "precio": 89.9,  "stock": 12},
    {"producto": "Mouse",    "precio": 45.5,  "stock": 30},
    {"producto": "Monitor",  "precio": 219.0, "stock": 7},
]

# pandas entiende la lista de diccionarios directamente
df = pd.DataFrame(catalogo)
print(df)
print()
print("Valor total del stock:", (df["precio"] * df["stock"]).sum())

# Y al reves: un DataFrame se serializa a JSON
print()
print(df.to_json(orient="records", indent=2, force_ascii=False))

---

## 3. Recursos, endpoints y verbos HTTP

En una API REST cada cosa es un **recurso** con una URL propia (**endpoint**):

```
https://api.ejemplo.com/users        ->  la colección de usuarios
https://api.ejemplo.com/users/42     ->  un usuario concreto
```

Y cada recurso acepta un conjunto reducido de operaciones, los **verbos**:

| Verbo | Qué hace | Cuando lo usamos |
|-------|----------|------------------|
| `GET` | Lee | Consultar el clima, listar pedidos |
| `POST` | Crea | Enviar un formulario, subir un registro |
| `PUT` | Reemplaza todo | Actualizar el título de un libro |
| `DELETE` | Borra | Cancelar un pedido |

La convención es importantísima: `GET` no debería modificar nada. Si tu `GET` borra registros,
cualquier crawler que pase por tu URL habrá borrado datos.

In [ ]:
VERBOS_HTTP = {
    "GET":    ("Lee un recurso",              "200 si existe / 404 si no"),
    "POST":   ("Crea un recurso nuevo",       "201 Created (+ cabecera Location) / 400"),
    "PUT":    ("Reemplaza un recurso entero", "200 si existe / 404 / 400"),
    "DELETE": ("Borra un recurso",            "204 No Content / 404"),
}

for verbo, (que, codigos) in VERBOS_HTTP.items():
    print(f"{verbo:7} {que:30} -> {codigos}")

El formato de intercambio puede ser JSON o XML. Los dos describen la misma persona;
JSON es el estándar actual porque es menos verboso:

In [ ]:
import xml.etree.ElementTree as ET

json_usuario = {"name": "John Doe", "email": "john.doe@example.com", "age": 30}

xml_usuario = """
    <user>
        <name>John Doe</name>
        <email>john.doe@example.com</email>
        <age>30</age>
    </user>
    """

raiz = ET.fromstring(xml_usuario)
print("Raiz:", raiz)

In [ ]:

xml_a_dict = {hijo.tag: hijo.text for hijo in raiz}

print("Desde JSON:", json_usuario)
print("Desde XML: ", xml_a_dict)
print("Coinciden: ", json_usuario == xml_a_dict)

# En XML los hijos pueden repetirse, y hay que agruparlos
print()
print("En XML un grupo repetido se agrupa a mano:")
catalogo_xml = """<catalogo>
    <producto><nombre>Teclado</nombre><precio>89.9</precio></producto>
    <producto><nombre>Mouse</nombre><precio>45.5</precio></producto>
</catalogo>"""
for hijo in ET.fromstring(catalogo_xml).findall("producto"):
    print("  ", {c.tag: c.text for c in hijo})

---

## 4. Códigos de estado: qué hacer con cada uno

El servidor **siempre** responde con un número de tres dígitos. Es lo primero que hay que
mirar: si no es 2xx, la respuesta no sirve aunque el cuerpo parezca bueno.

| Código | Significado | Tu script debería... |
|--------|-------------|----------------------|
| `200` | OK | procesar la respuesta |
| `201` | Created | procesar y, si hace falta, leer la cabecera `Location` |
| `204` | No Content | éxito sin cuerpo: no intentes parsear |
| `400` | Bad Request | corregir los parámetros y **no** reintentar igual |
| `401` | Unauthorized | mandar credenciales o corregir cómo se envían |
| `404` | Not Found | revisar la URL o el id |
| `429` | Too Many Requests | esperar y reintentar con más retraso |
| `500` | Server Error | el fallo es del servidor: reintentar con backoff |
| `503` | Service Unavailable | mantenimiento o saturación: esperar |

Regla práctica: **4xx es culpa del script, 5xx es culpa del servidor**. Se corrigen de forma
opuesta.

In [ ]:
ACCION_POR_CODIGO = {
    200: ("procesar la respuesta", False),
    201: ("procesar y leer Location", False),
    204: ("exito sin cuerpo", False),
    400: ("corregir los parametros", False),
    401: ("mandar credenciales", False),
    404: ("revisar la URL o el id", False),
    429: ("esperar y reintentar", True),
    500: ("reintentar con backoff", True),
    503: ("esperar: mantenimiento", True),
}

def que_hacer(codigo):
    # Devuelve la accion recomendada y si conviene reintentar.
    if codigo in ACCION_POR_CODIGO:
        return ACCION_POR_CODIGO[codigo]
    if 200 <= codigo < 300:
        return ("procesar la respuesta", False)
    if 400 <= codigo < 500:
        return ("corregir la peticion", False)
    return ("reintentar con backoff", True)

for codigo in (200, 404, 429, 500):
    accion, reintentar = que_hacer(codigo)
    print(f"{codigo}  reintentar={str(reintentar):5}  {accion}")

---

## 5. La librería requests: llamadas reales

`requests` es una biblioteca de terceros que hace legible lo que `urllib` hace en diez
líneas. Se instala aparte:

```bash
pip install requests
```

Su gran ventaja no es la sintaxis: es que **habla JSON**. `response.json()` devuelve un `dict`
de Python listo para usar, sin que tengas que pensar en la decodificación.

In [ ]:
import requests

API = "https://jsonplaceholder.typicode.com"

r = requests.get(f"{API}/todos/1", timeout=10)
r


In [ ]:
print(r.status_code, r.json())

Un objeto `Response` tiene más información de la que se ve a simple vista. Conocerla
ahorra la mitad de los `try/except` mal puestos.

In [ ]:
r = requests.get(f"{API}/todos/1", timeout=10)

print("status_code :", r.status_code)
print("ok          :", r.ok, "-> True si status_code < 400")
print("url         :", r.url)
print("elapsed     :", r.elapsed.total_seconds(), "s")
print("headers     :", r.headers["Content-Type"])
print("encoding    :", r.encoding)
print("text        :", r.text[:60], "...")
print("json()      :", r.json())
print("json() es el mismo dict cada vez:", r.json() == r.json())

# r.content son los bytes crudos: utiles para imagenes o archivos
print("content     :", r.content[:40], b"...")

`r.json()` es, por debajo, `json.loads(r.text)`. Handy cuando tenés el JSON de una fuente
que no usa `requests`, o cuando ya tenés el texto y no quieres perderlo.

In [ ]:
import json as json_modulo

r = requests.get(f"{API}/todos/1", timeout=10)

print("Iguales:", r.json() == json_modulo.loads(r.text))

# Y con un parametro:Pedir solo algunos campos
print("Solo los keys:", list(r.json().keys()))
print("Solo title:", r.json()["title"])


In [ ]:

# Pedir el JSON "crudo" (texto) cuando el servidor devuelve otra cosa
r_no_json = requests.get("https://httpbin.org/html", timeout=10)
print("\nEsta respuesta no es JSON, .json() falla:")

try:
    r_no_json.json()
except ValueError as e:
    print("  ValueError:", e)

In [ ]:
r_no_json.__dict__

### Ramificar según el código

Esto es el esqueleto que vas a repetir en todos tus scripts de API.

In [ ]:
def consultar_todo(todo_id):
    r = requests.get(f"{API}/todos/{todo_id}", timeout=10)

    if r.status_code == 200:
        return f"OK -> {r.json()['title']}"
    elif r.status_code == 404:
        return f"El todo {todo_id} no existe"
    elif r.status_code == 429:
        return "Demasiadas peticiones: hay que esperar"
    elif 500 <= r.status_code < 600:
        return "Fallo del servidor, se puede reintentar"
    else:
        return f"Codigo inesperado: {r.status_code}"

for identificador in (1, 2, 99999):
    print(f"todo {identificador:5}: {consultar_todo(identificador)}")

### El CRUD completo

Cuatro verbos contra la misma API. Fijate en que cada verbo usa su propio método de
`requests` y, cuando hay datos de entrada, los pasa en `json=`.

> JSONPlaceholder **no guarda nada**: responde con datos simulados. Sirve para practicar la
> estructura, no para construir sobre ella.

In [ ]:
# GET: listar
r = requests.get(f"{API}/users", timeout=10)
usuarios = r.json()
print(f"GET  /users      -> {r.status_code}, {len(usuarios)} usuarios")

# GET: un elemento concreto
r = requests.get(f"{API}/users/2", timeout=10)
print(f"GET  /users/2    -> {r.status_code}, {r.json()['name']}, {r.json()['email']}")

# POST: crear
nuevo = {"name": "Mora Diaz", "username": "morad", "email": "mora@example.com"}
r = requests.post(f"{API}/users", json=nuevo, timeout=10)
creado = r.json()
print(f"POST /users      -> {r.status_code}, id asignado: {creado['id']}")
print("Cabecera Location:", r.headers.get("Location", "(no la envia este servidor)"))

In [ ]:
r = requests.get(f"{API}/users", timeout=10)
usuarios = r.json()
print(f"GET  /users      -> {r.status_code}, {len(usuarios)} usuarios")

In [ ]:
# PUT: reemplazar
r = requests.put(f"{API}/users/2", json={"id": 2, "name": "Ervin Howell (editado)"}, timeout=10)
print(f"PUT  /users/2    -> {r.status_code}, {r.json()}")


In [ ]:
# PATCH: actualizar solo algunos campos
r = requests.patch(f"{API}/users/2", json={"username": "ervinh"}, timeout=10)
print(f"PATCH /users/2   -> {r.status_code}, username ahora: {r.json().get('username')}")


In [ ]:
# DELETE: borrar
r = requests.delete(f"{API}/users/2", timeout=10)
print(f"DELETE /users/2  -> {r.status_code}, cuerpo: {r.text!r}")


In [ ]:
# Ojo con la leccion: la guia dice que DELETE devuelve 204 No Content,
# y es lo habitual, pero este servidor concreto devuelve 200 con cuerpo vacio.
# Por eso nunca se debe asumir el codigo ni intentar parsear sin comprobarlo.

`raise_for_status()` es el atajo a toda la cadena de `if`. Lanza `HTTPError` si el código
es 4xx o 5xx, y no hace nada si fue bien. Combina perfecto con `try/except`.

In [ ]:
r = requests.get(f"{API}/todos/1", timeout=10)
r.raise_for_status()   # 200 -> no pasa nada
print("Todo bien:", r.json()["title"])

r = requests.get(f"{API}/todos/99999", timeout=10)
try:
    r.raise_for_status()
except requests.exceptions.HTTPError as e:
    print("\nHTTPError:", e)
    print("Codigo:", r.status_code)

---

## 6. timeout y parseo de XML

Sin `timeout`, `requests` espera **indefinidamente**. Si el servidor se cuelga, tu script
se queda colgado con él y no puedes ni cancelarlo con Ctrl+C.

`timeout` es el número de segundos que estamos dispuestos a esperar. Pasarlo **siempre** es
una buena práctica: convierte un cuelgue indefinido en un error que podés capturar.

In [ ]:
# Demostracion: un timeout tan corto que no da tiempo ni a la conexion
try:
    requests.get(f"{API}/todos/1", timeout=1e-6)
    print("La red respondio antes de agotar el timeout (conexion muy rapida)")
except requests.exceptions.Timeout as e:
    print("Timeout capturado:", type(e).__name__)
    print("-> Con timeout, esto es un error manejable en vez de un cuelgue.")

In [ ]:
# Y el caso complementario: ni timeout ni host valido
try:
    requests.get("https://dominio-inexistente.invalid/api", timeout=5)
    print("Se respondio (hay un proxy o cache en medio)")
except requests.exceptions.ConnectionError as e:
    print("ConnectionError capturado:", type(e).__name__)
    print("-> El dominio .invalid esta reservado por la RFC 2606 y nunca resuelve.")

Y el resto de XML, con el módulo `xml.etree.ElementTree` de la biblioteca estándar.
La estructura es jerárquica: se navega con `.find()`, `.findall()` y `.text`.

In [ ]:
import xml.etree.ElementTree as ET

catalogo_xml = """<?xml version="1.0" encoding="UTF-8"?>
<catalogo>
    <producto id="1"><nombre>Teclado</nombre><precio>89.90</precio><stock>12</stock></producto>
    <producto id="2"><nombre>Mouse</nombre><precio>45.50</precio><stock>30</stock></producto>
    <producto id="3"><nombre>Monitor</nombre><precio>219.00</precio><stock>7</stock></producto>
</catalogo>"""

raiz = ET.fromstring(catalogo_xml)
print("Etiqueta raiz:", raiz.tag)

# .text devuelve el contenido; si hay hijos, devuelve None
print("Precio del primero:", raiz.find("producto/precio").text)

# findall devuelve una lista de todos los que coinciden
for producto in raiz.findall("producto"):
    nombre = producto.find("nombre").text
    precio = float(producto.find("precio").text)
    stock = int(producto.find("stock").text)
    print(f"  {nombre:8} {precio:7.2f} x {stock:3} = {precio * stock:8.2f}")

# Busqueda por atributo
print()
print("El producto 3:", raiz.find("producto[@id='3']").find("nombre").text)

---

## 7. Postman para depurar

**Postman** es una aplicación de escritorio (gratis) para construir y probar peticiones HTTP
sin escribir código. Cada petición tiene cuatro cosas que llenar:

- **Método**: GET, POST, PUT, PATCH, DELETE, HEAD u OPTIONS
- **URL**
- **Headers** (encabezados)
- **Body** (cuerpo, cuando el método y el endpoint lo requieren)

Y la respuesta te muestra el código de estado, los encabezados y el cuerpo.

Además de los cuatro métodos más comunes, en el selector de Postman podés encontrar:

| Método | Para qué sirve | Cómo probarlo en Postman |
|--------|----------------|--------------------------|
| `PATCH` | Modifica solo algunos campos de un recurso. | Seleccionalo, indica el endpoint del recurso y envía en el body solo los campos que querés cambiar (normalmente como JSON). |
| `HEAD` | Pide los mismos encabezados que `GET`, pero sin recibir el cuerpo de la respuesta. | Seleccionalo e indica la URL; sirve para comprobar si un recurso existe o revisar encabezados sin descargar su contenido. |
| `OPTIONS` | Consulta qué métodos u opciones admite un endpoint; también se usa en comprobaciones CORS del navegador. | Seleccionalo e indica la URL; revisa los encabezados y la respuesta que envía el servidor. |

Postman también permite escribir un método personalizado con **Type a new method**. Solo
funcionará si la API lo reconoce; si no, el servidor puede responder `405 Method Not Allowed`.
El body y los encabezados dependen del método y de lo que documente cada API. Por ejemplo,
`HEAD` no devuelve cuerpo, mientras que `PATCH` suele llevar los cambios en el body.

Su valor real es el **método de diagnóstico** del módulo:

| Dónde falla | Dónde está el problema |
|-------------|------------------------|
| Falla en Postman | En la petición: URL mal escrita, datos incorrectos, falta la clave |
| **Funciona en Postman pero falla en Python** | En **tu código**: cómo construyes la petición o cómo lees la respuesta |

Esa segunda fila es la que ahorra horas. Antes de depurar Python, reproducí la petición a
mano.

In [ ]:
# Replica de una sesion tipica de Postman: GET a /todos/1
print("Peticion  ->  GET https://jsonplaceholder.typicode.com/todos/1")
print("Headers   ->  (ninguno, esta API es publica)")
print()

r = requests.get("https://jsonplaceholder.typicode.com/todos/1", timeout=10)

print("Respuesta")
print("  status  :", r.status_code)
print("  type    :", r.headers.get("Content-Type"))
print("  body    :", r.text[:120], "...")

In [ ]:
# Provocar el 404 a proposito, como en la demo del modulo
r = requests.get("https://jsonplaceholder.typicode.com/este-recurso-no-existe", timeout=10)
print("URL inexistente ->", r.status_code)
print("Cuerpo devuelto  ->", r.text[:120])

# Y con raise_for_status, el error salta con contexto claro
try:
    r.raise_for_status()
except requests.exceptions.HTTPError as e:
    print("\nHTTPError:", e)

Sobre las claves y el `401`: la causa más común no es que la clave esté mal, sino que
esté en el **sitio equivocado**. Cada API la espera en un lugar concreto y si la mandás en
otro, el servidor no la ve. Por eso conviene probar primero en Postman: ahí se cambia de
sección y se ve si el código de respuesta cambia.

In [ ]:
# Comparacion: la misma peticion segun donde vaya la clave
r_sin = requests.get("https://httpbin.org/bearer", timeout=10)
print("Sin cabecera Authorization ->", r_sin.status_code)

r_con = requests.get("https://httpbin.org/bearer", headers={"Authorization": "Bearer mi-token"}, timeout=10)
print("Con cabecera Authorization ->", r_con.status_code)

# Y con una clave en los parametros, como hacen muchas APIs meteorologicas
r_param = requests.get("https://httpbin.org/get", params={"key": "mi-clave-secreta"}, timeout=10)
print("Clave como parametro      ->", r_param.json()["args"])

---

## 8. Autenticación: API keys, OAuth y JWT

Dos conceptos que se mezclan todo el tiempo:

- **Autenticación** = quién eres. La analogía es mostrar el DNI en la puerta.
- **Autorización** = qué te dejan hacer una vez dentro.

| Mecanismo | Para qué sirve | Dónde va la credencial |
|-----------|-----------------|------------------------|
| **API key** | Identifica **a tu aplicación** | Cabecera o parámetro |
| **OAuth** | Acceso **delegado** por un usuario | Flujo con token |
| **JWT** | Transporta identidad y permisos **de forma verificable** | Cabecera `Authorization: Bearer` |

### API key

Es una llave digital. No verifica *quién es el usuario*: verifica *qué aplicación llama*. Su
seguridad es básica: si se filtra, cualquiera la usa.

**Nunca la escribas en el código.** Va en una variable de entorno.

In [ ]:
import os

# MAL: la clave viaja en el repositorio, en cada commit, para siempre
API_KEY = "a1b2c3d4e5f6-mi-clave-real"

# BIEN: se lee del entorno y, si no esta, el script avisa y para
API_KEY = os.environ.get("MI_API_KEY")

if not API_KEY:
    print("Falta la variable de entorno MI_API_KEY.")
    print("Definila con:  setx MI_API_KEY \"tu-clave\"   (Windows)")
    print("Y no la subas nunca al repositorio.")
else:
    print("Clave cargada, longitud:", len(API_KEY), "(nunca la imprimas entera)")

In [ ]:
# La misma llamada segun donde vaya la credencial
API_KEY = "demo-solo-para-prueba"

r = requests.get("https://httpbin.org/get", params={"key": API_KEY}, timeout=10)
print("Como parametro -> la API la ve en .json()['args']")
print (r.json())
print(r.json()["args"])


In [ ]:


r = requests.get("https://httpbin.org/get", headers={"X-Api-Key": API_KEY}, timeout=10)
print("Como cabecera  -> la API la ve en .json()['headers']")
print (r.json())
print(r.json()["headers"].get("X-Api-Key", "(no la envio)"))

### OAuth

OAuth resuelve el problema de **acceso delegado**: el usuario autoriza a tu aplicación sin
cederle su contraseña. Es el mecanismo detrás de "Continuar con Google" o de dar acceso a
OneDrive.

El flujo resumido:

1. Tu app manda al usuario a la página del proveedor
2. El usuario autoriza ahí, con su contraseña, que nunca pasa por tu servidor
3. El proveedor devuelve un **access token** con alcance limitado
4. Cuando caduca, se canjea por uno nuevo con un **refresh token**

Para leer los datos del usuario basta con el access token. La librería habitual es
`requests_oauthlib`, o el SDK del propio proveedor.

### JWT

Un **JSON Web Token** es un sobre digital con tres partes separadas por puntos:

```
eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9 . eyJ1c2VyX2lkIjoxMjN9 . 4pcPyMD09olPSyXn...
└──────── cabecera ─────────────────┘ └────── payload ─────┘ └──── firma ────┘
```

- La **cabecera** dice con qué algoritmo se firmó
- El **payload** lleva los *claims*: `user_id`, `roles`, `exp`
- La **firma** es lo que impide que alguien altere el contenido

Es **sin estado**: el servidor no guarda nada, solo verifica la firma.

In [ ]:
import jwt
import time

# Clave de ejemplo: en produccion va en el entorno y es mucho mas larga
CLAVE_SECRETA = "clave-de-demostracion-de-32-caracteres-minimo"

claims = {
    "user_id": 123,
    "nombre": "Ana Lopez",
    "roles": ["admin", "reportes"],
    "exp": int(time.time()) + 3600,   # caduca en 1 hora
}

token = jwt.encode(claims, CLAVE_SECRETA, algorithm="HS256")
print("Token:", token[:150], "...")

partes = token.split(".")
print("\nPartes del token:", len(partes))
print("Cabecera :", jwt.get_unverified_header(token))
print("Payload :", jwt.decode(token, CLAVE_SECRETA, algorithms=["HS256"]))

In [ ]:
# Verificar de verdad: la firma protege contra manipulacion
# Forjamos un token con los mismos roles cambiados a "admin" pero la firma original
cabecera, _, firma = token.split(".")
claims_falsos = {**claims, "roles": ["admin"]}
payload_falso = jwt.encode(claims_falsos, "otra-clave--distinta-de-32-caracteres-x", algorithm="HS256")
token_forjado = f"{cabecera}.{payload_falso.split('.')[1]}.{firma}"

print("El contenido dice:", jwt.decode(token_forjado, CLAVE_SECRETA,
      algorithms=["HS256"], options={"verify_signature": False})["roles"])
try:
    jwt.decode(token_forjado, CLAVE_SECRETA, algorithms=["HS256"])
except jwt.InvalidSignatureError as e:
    print("InvalidSignatureError:", e, "-> alguien manipulo el contenido")

# Y la expiracion, que es el claim exp
token_vencido = jwt.encode({"user_id": 1, "exp": int(time.time()) - 10}, CLAVE_SECRETA, algorithm="HS256")
try:
    jwt.decode(token_vencido, CLAVE_SECRETA, algorithms=["HS256"])
except jwt.ExpiredSignatureError as e:
    print("ExpiredSignatureError:", e, "-> hay que pedir un token nuevo")

In [ ]:
# El token viaja en la cabecera Authorization
r = requests.get("https://httpbin.org/bearer", headers={"Authorization": f"Bearer {token}"}, timeout=10)
print("El servidor recibio el token:", r.json().get("token", "(no llego)")[:30], "...")

# Y ahi se comprueba el alcance
payload = jwt.decode(token, CLAVE_SECRETA, algorithms=["HS256"])
print("¿Puede este usuario generar informes?", "reportes" in payload["roles"])

### Autorización: RBAC frente a CBAC

Una vez autenticado, hay que decidir qué puede hacer cada uno:

- **RBAC** (*Role-Based Access Control*): los permisos dependen del **rol**.
- **CBAC** (*Claims-Based Access Control*): los permisos dependen de los **atributos** que
  viajan en el token (departamento, nivel, antigüedad).

In [ ]:
PERMISOS = {
    "admin":   {"leer", "escribir", "borrar", "facturar"},
    "editor":  {"leer", "escribir"},
    "lector":  {"leer"},
    "invitado": set(),
}

def puede(rol, accion):
    return accion in PERMISOS.get(rol, set())

print("Permisos por rol:")
for rol, acciones in PERMISOS.items():
    print(f"  {rol:8} {sorted(acciones)}")

print()
for rol in ("admin", "editor", "lector", "invitado"):
    print(f"{rol:8} ¿puede borrar? {str(puede(rol, 'borrar')):5} ¿puede leer? {puede(rol, 'leer')}")

# En CBAC la decision se toma sobre los claims, no sobre un rol fijo
claims = {"user": "Mora", "departamento": "ventas", "nivel": 3}
print()
print("CBAC: ¿esta persona de", claims["departamento"], "puede ver los reportes financieros?")
print("  ", claims["departamento"] == "finanzas" or claims["nivel"] >= 5)

---

## 9. Rate limiting y gestión de errores

### Límites de velocidad

Toda API pública tiene un tope de peticiones por minuto u hora. Excederlo trae un bloqueo
temporal o permanente. Existen por tres motivos:

1. **Evitar la sobrecarga** del servidor
2. **Uso justo**: que un usuario no monopolice el servicio
3. **Controlar costes**, cuando la API se factura por uso

La recomendación siempre es la misma: **leé la documentación de la API** antes de escribir
el código, porque los límites cambian sin aviso.

Las cuatro técnicas del módulo:

1. **`time.sleep()`** entre llamadas
2. **Caché** para los datos que se piden mucho
3. **Reintentos** para problemas temporales
4. **Upgrade de plan**, si el límite se queda corto

In [ ]:
import time

# 4 peticiones a la API con 1 segundo de pausa entre cada una
INICIO = time.perf_counter()

for numero in range(1, 5):
    print(requests.get(f"{API}/todos/{numero}", timeout=10))
    print(f"Peticion {numero} hecha")
    time.sleep(1)   # <- la pausa que respeta el limite

print(f"Tardamos {time.perf_counter() - INICIO:.1f} s en 4 peticiones")

### Caché

Si llamás a la misma URL muchas veces, la respuesta no cambia entre llamadas. `lru_cache`
guarda el resultado en memoria y hace **una sola llamada** por cada argumento distinto.

In [ ]:
from functools import lru_cache

llamadas_reales = 0

@lru_cache(maxsize=32)
def obtener_usuario(user_id):
    # Se ejecuta una sola vez por cada user_id, no una por peticion.
    global llamadas_reales
    llamadas_reales += 1
    r = requests.get(f"{API}/users/{user_id}", timeout=10)
    return r.json()

for _ in range(3):
    usuario = obtener_usuario(1)   # 3 veces
print("Nombre:", usuario["name"])
print("Llamadas HTTP reales:", llamadas_reales, "-> las otras 2 salieron de la cache")

# El mismo id en la cache, uno nuevo no
obtener_usuario(2)
print("Tras pedir el usuario 2:", llamadas_reales)

# Como liberarla
obtener_usuario.cache_clear()
obtener_usuario(1)
print("Tras limpiar la cache:", llamadas_reales)

### Reintentos con backoff exponencial

Si el error es temporal, repetir tiene sentido. Pero repetir **inmediatamente** reproduce el
mismo resultado: lo que hay que hacer es esperar cada vez un poco más.

`2 ** intento` da 1 s, 2 s, 4 s, 8 s...

> **Solo para 5xx y 429.** Reintentar un `400` o un `404` no sirve de nada: la petición es
> incorrecta y va a seguir siéndolo. Reintentar también agota los límites de la API.

In [ ]:
def con_reintentos(url, intentos=4, **kwargs):
    # Llama a `url` reintentando ante fallos temporales, con espera creciente.
    for intento in range(intentos):
        try:
            r = requests.get(url, timeout=10, **kwargs)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            print(f"  intento {intento + 1}: {type(e).__name__}, reintentando")
            time.sleep(2 ** intento)
            continue

        if r.status_code == 429 or 500 <= r.status_code < 600:
            print(f"  intento {intento + 1}: {r.status_code}, temporal, reintentando")
            time.sleep(2 ** intento)
            continue

        try:
            r.raise_for_status()
        except requests.exceptions.HTTPError as e:
            # 4xx: la peticion esta mal. Reintentar es tirar el tiempo.
            print(f"  {r.status_code} no es reintentable: {e.response.status_code}")
            return None

        return r

    print("  se agotaron los intentos")
    return None

print("URL valida:")
r = con_reintentos(f"{API}/todos/1")
print("  resultado:", r.status_code if r else None)

print("\nURL que devuelve 404 (no se reintenta, no es temporal):")
r = con_reintentos(f"{API}/todo-que-no-existe")
print("  resultado:", r.status_code if r else None)

### Los cuatro tipos de error

| Tipo | Ejemplos | De quién es el problema |
|------|----------|------------------------|
| **4xx** | 400, 401, 404, 429 | Tu script: corregir la petición |
| **5xx** | 500, 503 | El servidor: esperar y reintentar |
| **Conectividad** | `ConnectionError`, `Timeout` | La red: reintentar con espera |
| **Datos** | `KeyError`, `ValueError`, `JSONDecodeError` | La respuesta: validar antes de usar |

Los dos últimos son los que se olvidan: una respuesta con formato inesperado rompe el script
igual que un 500.

In [ ]:
def analizar_respuesta(url):
    try:
        r = requests.get(url, timeout=10)
    except requests.exceptions.Timeout:
        return "TIMEOUT: el servidor no respondio a tiempo"
    except requests.exceptions.ConnectionError:
        return "CONEXION: no se pudo ni conectar"

    if r.status_code == 404:
        return "404: la URL no existe"
    if r.status_code == 429:
        return "429: demasiadas peticiones, esperar"
    if 500 <= r.status_code < 600:
        return f"{r.status_code}: fallo del servidor, reintentable"
    if not r.ok:
        return f"{r.status_code}: revisar la peticion"

    # Status 200, pero todavia falta comprobar que el cuerpo sea lo esperado
    try:
        datos = r.json()
    except ValueError:
        return "200: la respuesta no es JSON"

    return f"200: OK, claves esperadas = {sorted(datos)[:3]}..."

print(analizar_respuesta(f"{API}/todos/1"))
print(analizar_respuesta(f"{API}/no-existe"))
print(analizar_respuesta("https://dominio-inexistente.invalid/x"))
print(analizar_respuesta("https://httpbin.org/html"))

### `logging`: dejar rastro

En un script que corre solo a las 3 AM no hay nadie mirando la consola. `logging` escribe
marcas de tiempo y contexto a un archivo, y es lo que te va a decir qué pasó cuando la cosa
 falle de verdad.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("modulo3")

def tarea_con_log(descripcion, funcion):
    logger.info("Iniciando: %s", descripcion)
    try:
        resultado = funcion()
        logger.info("OK: %s -> %s", descripcion, resultado)
        return resultado
    except Exception:
        # logger.exception() añade el traceback completo al registro
        logger.exception("FALLO en: %s", descripcion)
        return None

tarea_con_log("descargar todo 1", lambda: requests.get(f"{API}/todos/1", timeout=10).json()["title"])
tarea_con_log("descargar todo 99999", lambda: requests.get(f"{API}/todos/99999", timeout=10).json()["title"])

print("\nLas dos lineas anteriores salieron del logger, con su hora.")

### Los cuatro mensajes de error que importan

1. **Bloques `try/except`** alrededor de las llamadas
2. **Mensajes informativos**: qué se intentó hacer y cómo arreglarlo, sin volcar datos internos
3. **Reintentos** para lo que sea temporal
4. **`logging`** para que quede constancia cuando no haya nadie mirando

---

# Quiz interactivo: APIs, requests y autenticación

Responde cada pregunta y pulsa **Ver respuesta**. Hay preguntas de una sola opcion y de multiple respuesta (selecciona todas las que correspondan). Si fallas, lee la explicacion: ahi esta el porque.

In [1]:
# --- Configuracion del quiz ---
# 1) Localizar la raiz del proyecto (la que contiene la carpeta 'quiz') y anadirla al sys.path
import sys
from pathlib import Path

RAIZ = Path.cwd()
while RAIZ.parent != RAIZ and not (RAIZ / "quiz").exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.append(str(RAIZ))

# 2) Importar funciones del modulo reutilizable
from quiz.quiz import aplicar_estilos, crear_quiz, crear_quiz_multiple

# 3) Aplicar el CSS (legible en VS Code y JupyterLab)
aplicar_estilos()

# 4) Cargar respuestas y explicaciones desde el JSON (misma carpeta que el notebook)
import json
with open(Path.cwd() / "Modulo_3_APIs_preguntas.json", encoding="utf-8") as fh:
    respuestas = json.load(fh)

print("Quiz listo:", ", ".join(respuestas.keys()))

Quiz listo: q1, q2, q3, q4, q5, q6, q7, q8


### Pregunta 1

Estás desarrollando una aplicación para dispositivos móviles que ayuda a los usuarios a planificar excursiones de senderismo. Quieres integrar información meteorológica en tiempo real y mapas interactivos. ¿Qué tipo de tecnología permitiría mejor esta integración?

- Protocolo HTTP
- GraphQL
- Interfaces de programación de aplicaciones (API)
- SOAP

In [2]:
q1_data = respuestas["q1"]
crear_quiz(q1_data["opciones"], q1_data["correcta"], q1_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) Protocolo HTTP', 'A'), ('B) GraphQL', 'B'), ('C) Inter…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 2

Estás diseñando una aplicación web que permite a los usuarios acceder a sus datos financieros desde varias cuentas bancarias. ¿Qué combinación de métodos sería la más adecuada para autenticar y autorizar de forma segura a los usuarios y aplicaciones que accedan a tu API?

- Claves API para la identificación inicial y la limitación de la tasa
- Autenticación abierta (OAuth) para que los usuarios puedan acceder a sus datos sin compartir sus contraseñas
- Contraseñas en texto plano almacenadas en una base de datos
- Web tokens (JWT) para transmitir de forma segura la información del usuario y los permisos entre tu aplicación y la API

In [3]:
q2_data = respuestas["q2"]
crear_quiz_multiple(q2_data["opciones"], q2_data["correctas"], q2_data["explicacion"])

Checkbox(value=False, description='A) Claves API para la identificación inicial y la limitación de la tasa', i…

Checkbox(value=False, description='B) Autenticación abierta (OAuth) para que los usuarios puedan acceder a sus…

Checkbox(value=False, description='C) Contraseñas en texto plano almacenadas en una base de datos', indent=Fal…

Checkbox(value=False, description='D) Web tokens (JWT) para transmitir de forma segura la información del usua…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 3

Estás trabajando con una API que devuelve datos en formato JSON y necesitas procesarlos dentro de tu script Python. ¿Cuál de las siguientes opciones describe mejor la relación entre JSON y los diccionarios de Python?

- Los diccionarios JSON y Python son formatos de datos completamente ajenos.
- Los diccionarios Python son un tipo específico de objeto JSON utilizado únicamente para el desarrollo web.
- JSON es un formato legible por humanos que puede convertirse fácilmente en diccionarios de Python para su manipulación.
- JSON sólo puede procesarse mediante bibliotecas especializadas de JavaScript.

In [4]:
q3_data = respuestas["q3"]
crear_quiz(q3_data["opciones"], q3_data["correcta"], q3_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) Los diccionarios JSON y Python son formatos de datos c…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 4

Estás creando una aplicación que interactúa con una API meteorológica para recuperar y mostrar datos. Necesitas obtener la temperatura actual de una ciudad concreta. ¿Cómo estructurarías tu petición a la API para conseguirlo?

- Envíe una solicitud PUT al punto final /weather con el nombre de la ciudad en el cuerpo de la solicitud.
- Envíe una solicitud GET al punto final /weather/{city_name}.
- Envíe una solicitud POST al punto final /weather con el nombre de la ciudad en el cuerpo de la solicitud.
- Envíe una solicitud DELETE al punto final /weather/{city_name}.

In [5]:
q4_data = respuestas["q4"]
crear_quiz(q4_data["opciones"], q4_data["correcta"], q4_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) Envíe una solicitud PUT al punto final /weather con el…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 5

Estás desarrollando una aplicación Python que necesita recuperar datos de un servidor remoto utilizando una API. Quieres elegir una librería que simplifique el proceso de hacer llamadas a la API y manejar las respuestas. ¿Qué biblioteca sería la más adecuada para esta tarea?

- requests
- json
- socket
- xml

In [6]:
q5_data = respuestas["q5"]
crear_quiz(q5_data["opciones"], q5_data["correcta"], q5_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) requests', 'A'), ('B) json', 'B'), ('C) socket', 'C'),…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 6

Estás creando una aplicación web que permite a los usuarios acceder a sus archivos privados almacenados en un servicio en la nube. Quieres permitir que los usuarios concedan a tu aplicación acceso a sus archivos sin compartir sus contraseñas del servicio en la nube. ¿Qué mecanismo sería el más adecuado?

- Claves API
- Autenticación básica HTTP
- JWT
- OAuth

In [7]:
q6_data = respuestas["q6"]
crear_quiz(q6_data["opciones"], q6_data["correcta"], q6_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) Claves API', 'A'), ('B) Autenticación básica HTTP', 'B…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 7

Estás utilizando Postman para solucionar una solicitud de API que devuelve un error 401 Unauthorized. Has confirmado que tu clave de API es correcta. ¿Cuál es la causa más probable de este error?

- La URL del punto final de la API es incorrecta.
- El servidor API está experimentando un error interno.
- La clave API falta o es incorrecta en la solicitud de Postman.
- La clave API no se está enviando en la ubicación correcta dentro de la solicitud (por ejemplo, cabeceras, parámetros).

In [8]:
q7_data = respuestas["q7"]
crear_quiz(q7_data["opciones"], q7_data["correcta"], q7_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) La URL del punto final de la API es incorrecta.', 'A')…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 8

Estás escribiendo un script que interactúa con una API de redes sociales con un límite de 100 peticiones por minuto. ¿Qué técnicas son adecuadas para gestionar este límite y garantizar que el script se ejecuta sin problemas?

- Utilice la función time.sleep() en Python para pausar su script entre peticiones.
- Implemente el tratamiento de errores con bloques try-except para capturar las excepciones de límite de velocidad y reintentar las peticiones tras un retardo.
- Introduzca un retardo fijo de 0,5 segundos entre cada solicitud de API.
- Almacene en caché los resultados de las llamadas frecuentes a la API para evitar peticiones redundantes.

In [9]:
q8_data = respuestas["q8"]
crear_quiz_multiple(q8_data["opciones"], q8_data["correctas"], q8_data["explicacion"])

Checkbox(value=False, description='A) Utilice la función time.sleep() en Python para pausar su script entre pe…

Checkbox(value=False, description='B) Implemente el tratamiento de errores con bloques try-except para captura…

Checkbox(value=False, description='C) Introduzca un retardo fijo de 0,5 segundos entre cada solicitud de API.'…

Checkbox(value=False, description='D) Almacene en caché los resultados de las llamadas frecuentes a la API par…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

## ¡Fin del quiz!

Si fallaste alguna, releé la explicación: ahí está el porqué de cada respuesta. En particular,
fíjate en la diferencia entre **autenticar** y **autorizar**, y en por qué un reintento con
`time.sleep(0.5)` fijo no sustituye a leer la documentación de la API.